# TJES - Tribunal de Justica do Espirito Santo

Example notebook for the TJES jurisprudence scraper (`cjsg`).

The TJES scraper queries the public API at `sistemas.tjes.jus.br/consulta-jurisprudencia/` and supports five Solr cores:

| Core | Description |
|------|-------------|
| `pje1g` | First instance (PJe) |
| `pje2g` | Second instance (PJe) — **default** |
| `pje2g_mono` | Second instance monocratic decisions (PJe) |
| `legado` | Legacy second instance |
| `turma_recursal_legado` | Legacy appellate panels (Projudi) |

## Basic search

In [1]:
import juscraper as jus

# Create a TJES scraper
tjes = jus.scraper('tjes')

In [2]:
# Search jurisprudence (default core: pje2g)
dados_cjsg = tjes.cjsg('dano moral', paginas=range(1, 3))

print(dados_cjsg.shape)
dados_cjsg.head(3)

Baixando paginas TJES: 100%|██████████| 2/2 [00:01<00:00,  1.32it/s]

(40, 23)


,nr_processo,ementa,magistrado,orgao_julgador,classe_judicial,classe_judicial_sigla,assunto_principal,jurisdicao,competencia,dt_juntada,...,localizacao,cargo_julgador,cd_assunto_principal,cd_classe_judicial,id_assunto_principal,id_classe_judicial,id_jurisdicao,id_localizacao,id_cargo_julgador,id_bin
0,5000333-79.2017.8.08.0030,\nVoto servindo como ementa.\n,GRACIENE PEREIRA PINTO,Turma Recursal - 4ª Turma,Recurso Inominado Cível,RecInoCiv,Indenização por Dano Moral,Turma Recursal,Turmas Recursais,2020-03-12,...,4ª TURMA RECURSAL GABINETE 3,Juiz Membro,7779,460,622,321,2,66.0,57,393003
1,5002610-68.2017.8.08.0030,\nVoto servindo como ementa.\n,MARCELO MATTAR COUTINHO,Turma Recursal - 4ª Turma,Recurso Inominado Cível,RecInoCiv,Fornecimento de Energia Elétrica,Turma Recursal,Turmas Recursais,2020-06-16,...,4ª TURMA RECURSAL GABINETE 1,Juiz Membro,7760,460,602,321,2,63.0,69,474881
2,5011718-32.2023.8.08.0024,\n \n\n\nEMENTA: RECURSO INOMINADO. AÇÃO DE IN...,PAULO ABIGUENEM ABIB,Turma Recursal - 1ª Turma,RECURSO INOMINADO CÍVEL,RecInoCiv,Indenização por Dano Moral,Turma Recursal,Turmas Recursais - Cível,2025-08-29,...,1ª TURMA RECURSAL GABINETE 2,Juiz Membro,10433,460,588,321,2,52.0,67,14355858


In [3]:
# Available columns
dados_cjsg.columns.tolist()

['nr_processo',
 'ementa',
 'magistrado',
 'orgao_julgador',
 'classe_judicial',
 'classe_judicial_sigla',
 'assunto_principal',
 'jurisdicao',
 'competencia',
 'dt_juntada',
 'id',
 'acordao',
 'lista_assunto',
 'localizacao',
 'cargo_julgador',
 'cd_assunto_principal',
 'cd_classe_judicial',
 'id_assunto_principal',
 'id_classe_judicial',
 'id_jurisdicao',
 'id_localizacao',
 'id_cargo_julgador',
 'id_bin']

In [4]:
# Preview ementa
print(dados_cjsg['ementa'].iloc[0][:300])


Voto servindo como ementa.



## Using filters

The TJES scraper supports several filters: `magistrado`, `orgao_julgador`, `classe_judicial`, `jurisdicao`, `assunto`, date range (`data_julgamento_inicio` / `data_julgamento_fim`), `busca_exata`, and `ordenacao`.

In [5]:
# Filter by date range and magistrado
dados_filtrados = tjes.cjsg(
    'direito',
    data_julgamento_inicio='2024-01-01',
    data_julgamento_fim='2024-06-30',
    magistrado='HELIMAR PINTO',
    paginas=1,
)

print(dados_filtrados.shape)
dados_filtrados[['nr_processo', 'magistrado', 'classe_judicial', 'dt_juntada']].head(5)

Baixando paginas TJES: 100%|██████████| 1/1 [00:00<00:00, 10.22it/s]

(20, 23)


,nr_processo,magistrado,classe_judicial,dt_juntada
0,5005361-74.2024.8.08.0000,HELIMAR PINTO,AGRAVO DE EXECUÇÃO PENAL,2024-06-25
1,0001362-38.2023.8.08.0000,HELIMAR PINTO,MANDADO DE SEGURANÇA CÍVEL,2024-02-29
2,5003125-52.2024.8.08.0000,HELIMAR PINTO,HABEAS CORPUS CRIMINAL,2024-04-18
3,0026004-72.2016.8.08.0048,HELIMAR PINTO,APELAÇÃO CRIMINAL,2024-04-25
4,5005989-63.2024.8.08.0000,HELIMAR PINTO,HABEAS CORPUS CRIMINAL,2024-06-25


## Querying different cores

By default, the scraper queries `pje2g` (second instance). You can switch cores with the `core` parameter.

In [6]:
# Search monocratic decisions (2nd instance)
dados_mono = tjes.cjsg('dano moral', core='pje2g_mono', paginas=1)

print(f"Monocratic results: {len(dados_mono)} rows")
dados_mono[['nr_processo', 'magistrado', 'classe_judicial']].head(3)

Baixando paginas TJES: 100%|██████████| 1/1 [00:00<00:00, 10.46it/s]

Monocratic results: 20 rows


,nr_processo,magistrado,classe_judicial
0,5001274-40.2022.8.08.0002,GRECIO NOGUEIRA GREGIO,Recurso Inominado Cível
1,5023551-77.2024.8.08.0035,DOUGLAS DEMONER FIGUEIREDO,RECURSO INOMINADO CÍVEL
2,5001473-59.2023.8.08.0024,SALOMAO AKHNATON ZOROASTRO SPENCER ELESBON,Recurso Inominado Cível


## First instance (cjpg)

Use `cjpg()` to query first-instance decisions (core `pje1g`). It accepts the same filters as `cjsg` (except `core`).

Note: `cjsg()` does **not** accept `core="pje1g"` — use `cjpg()` instead.

In [7]:
# Search first-instance decisions
dados_1g = tjes.cjpg('dano moral', paginas=1)

print(f"First instance results: {len(dados_1g)} rows")
dados_1g[['nr_processo', 'magistrado', 'classe_judicial', 'dt_juntada']].head(5)

Baixando paginas TJES: 100%|██████████| 1/1 [00:00<00:00,  5.04it/s]

First instance results: 20 rows


,nr_processo,magistrado,classe_judicial,dt_juntada
0,5000704-46.2022.8.08.0037,MARCELO MATTAR COUTINHO,Procedimento do Juizado Especial Cível,2023-09-28
1,5000057-89.2018.8.08.0005,EVANDRO COELHO DE LIMA,Procedimento do Juizado Especial Cível,2019-12-17
2,5004066-91.2024.8.08.0035,INES VELLO CORREA,Procedimento do Juizado Especial Cível,2024-11-08
3,5000045-13.2017.8.08.0037,MARCELO MATTAR COUTINHO,Procedimento do Juizado Especial Cível,2020-01-04
4,5005643-41.2022.8.08.0014,SALOMAO AKHNATON ZOROASTRO SPENCER ELESBON,Procedimento do Juizado Especial Cível,2022-11-01


## Download and parse separately

For more control, you can use `cjsg_download` and `cjsg_parse` separately. This is useful when you want to cache the raw responses or inspect them before parsing.

In [8]:
# Download raw JSON responses
brutos = tjes.cjsg_download('direito', paginas=1)

# Inspect raw response structure
print(f"Pages downloaded: {len(brutos)}")
print(f"Keys in response: {list(brutos[0].keys())}")
print(f"Total results available: {brutos[0]['total']}")
print(f"Docs in this page: {len(brutos[0]['docs'])}")

Baixando paginas TJES: 100%|██████████| 1/1 [00:00<00:00,  6.54it/s]

Pages downloaded: 1
Keys in response: ['core_used', 'docs', 'facets', 'page', 'per_page', 'total', 'total_pages']
Total results available: 167559
Docs in this page: 20


In [9]:
# Parse into DataFrame
df = tjes.cjsg_parse(brutos)
df.head(3)

,nr_processo,ementa,magistrado,orgao_julgador,classe_judicial,classe_judicial_sigla,assunto_principal,jurisdicao,competencia,dt_juntada,...,localizacao,cargo_julgador,cd_assunto_principal,cd_classe_judicial,id_assunto_principal,id_classe_judicial,id_jurisdicao,id_localizacao,id_cargo_julgador,id_bin
0,5002933-85.2025.8.08.0000,\nEmenta: DIREITO DAS SUCESSÕES. AGRAVO DE INS...,ARTHUR JOSE NEIVA DE ALMEIDA,4ª Câmara Cível,AGRAVO DE INSTRUMENTO,AI,Inventário e Partilha,Tribunal de Justiça,Cível Isolada,2025-06-13,...,016 - Gabinete Des. ARTHUR JOSÉ NEIVA DE ALMEIDA,Desembargador(a),7687,202,1636,238,1,36,117,13176473
1,5007013-97.2022.8.08.0000,\nDIREITO PROCESSUAL CIVIL. AGRAVO DE INSTRUME...,JOSE PAULO CALMON NOGUEIRA DA GAMA,2ª Câmara Cível,AGRAVO DE INSTRUMENTO,AI,Promessa de Compra e Venda,Tribunal de Justiça,Cível Isolada,2023-08-24,...,005 - Gabinete Des. JOSÉ PAULO CALMON NOGUEIRA...,Desembargador(a),10496,202,2087,238,1,21,149,5253374
2,5002710-35.2025.8.08.0000,\nEmenta: DIREITO CIVIL E PROCESSUAL CIVIL. AG...,ARTHUR JOSE NEIVA DE ALMEIDA,4ª Câmara Cível,AGRAVO DE INSTRUMENTO,AI,Inventário e Partilha,Tribunal de Justiça,Cível Isolada,2025-10-03,...,Gabinete Des. ARTHUR JOSÉ NEIVA DE ALMEIDA,Desembargador(a),7687,202,1636,238,1,36,117,14653808
